# This file generates cn data (for a single strike) and save to './data/{filename}.parquet'

### Info

There are 7 versions, v1-v7, of the crank nicolson implementation for eu options. Each is slightly more optimised than the other. v4-v7 have 1 step of rannacher smoothing. None have Richardson extrapolation yet.

**european versions**: src.options.EuropeanOption.price_CN(version =...)

 - "v1":   initial version made: unoptimised & stores V for all t_steps. Uses scipy's splu(L) for LU factorisation

 - "v2":    modified v1 to only store the most recent t_step. Still uses splu(L).

 - "v3":    modified v2 to use scipy's LAPACK for tridiagonal solve instead of scipy's splu. Also no longer define R.

 - "v4":     modified v3 to update V in-place, and also introduce Rannacher smoothing. Still uses LAPACK.

 - "v5":    modified v4 to use custom tridiagonal solve that can be JIT compiled using numba.

 - "v6":    modified v5 to JIT compile the whole cn loop, not just tridiagonal solve.

 - "v7":    modified v6 to compute d_inverse outside the loop and use  multiplication by d_inverse during back substitution instead

**american versions**: src.options.AmericanOption.price_CN(version =...)

- "v1":    essentially the v7 implementation for european options but checks for early exercise (no dividends)




In [18]:
import pandas as pd
from src.sweep import sweep, run, run_v2, cn_european, cn_american, mc_european


# generate european style call option data using crank nicolson

In [19]:
configs = sweep(
                K=100, T=1.0, contract_type="call", S0=100, r=0.05, sigma=0.2,
                s_steps=range(32,2000,128), t_steps=range(32,3000,128),
                version=("v1", "v2", "v3", "v4", "v5", "v6", "v7")
)


df_cn_eu = run(configs=configs, fn=cn_european, n_reps=1, save_dir=None)

Running:   0%|          | 0/2688 [00:00<?, ?run/s]

# generate european style call option data using monte carlo

In [20]:
base = dict(K=100, T=1.0, contract_type="call", S0=100, r=0.05, sigma=0.2,
              n=(1000, 2000, 4000), seed=range(10))

discretised_configs = sweep(**base, steps=(1000,1000), generator=('gbm_paths_euler', 'gbm_paths_milstein'))
exact_configs = sweep(**base, generator=('gbm_exact_integration'))

df_mc_eu = run(discretised_configs + exact_configs, mc_european, n_reps=1)

Running:   0%|          | 0/150 [00:00<?, ?run/s]

# generate american style call and put option data using crank nicolson

In [21]:
configs = sweep(
                K=100, T=1.0, contract_type=("call", "put"), S0=100, r=0.05, sigma=0.2,
                s_steps=range(32,2000,128), t_steps=range(32,3000,128),
                version=("v1"),
)

df_cn_am = run(configs, cn_american, n_reps=1)


Running:   0%|          | 0/768 [00:00<?, ?run/s]

In [24]:
df = pd.concat([df_cn_eu, df_cn_am, df_mc_eu],
               ignore_index=True)

In [25]:
from pathlib import Path

save_to = Path("./data/test_run.parquet")
df.to_parquet(save_to)
display(df.head(10))

,K,T,contract_type,S0,r,sigma,s_steps,t_steps,version,rep,method,result,time_ms,started,n,seed,steps,generator
0,100,1.0,call,100,0.05,0.2,1184.0,1824.0,v2,0,cn_european,10.450511,35.8298,2026-09-18 11:28:09.499482,NaN,NaN,NaN,NaN
1,100,1.0,call,100,0.05,0.2,1184.0,800.0,v6,0,cn_european,10.450511,5.5591,2026-09-18 11:28:09.535346,NaN,NaN,NaN,NaN
2,100,1.0,call,100,0.05,0.2,1056.0,1184.0,v7,0,cn_european,10.450492,2.8240,2026-09-18 11:28:09.540945,NaN,NaN,NaN,NaN
3,100,1.0,call,100,0.05,0.2,1312.0,2976.0,v4,0,cn_european,10.450525,37.0788,2026-09-18 11:28:09.543793,NaN,NaN,NaN,NaN
4,100,1.0,call,100,0.05,0.2,288.0,416.0,v7,0,cn_european,10.449358,0.3751,2026-09-18 11:28:09.580899,NaN,NaN,NaN,NaN
5,100,1.0,call,100,0.05,0.2,416.0,2464.0,v7,0,cn_european,10.449997,2.2033,2026-09-18 11:28:09.581288,NaN,NaN,NaN,NaN
6,100,1.0,call,100,0.05,0.2,288.0,32.0,v6,0,cn_european,10.449101,0.0922,2026-09-18 11:28:09.583497,NaN,NaN,NaN,NaN
7,100,1.0,call,100,0.05,0.2,160.0,288.0,v3,0,cn_european,10.446620,1.3009,2026-09-18 11:28:09.583593,NaN,NaN,NaN,NaN
8,100,1.0,call,100,0.05,0.2,1056.0,288.0,v3,0,cn_european,10.450496,3.6659,2026-09-18 11:28:09.584897,NaN,NaN,NaN,NaN
9,100,1.0,call,100,0.05,0.2,160.0,1568.0,v2,0,cn_european,10.446617,10.3654,2026-09-18 11:28:09.588578,NaN,NaN,NaN,NaN


In [ ]:
"""
import numpy as np
vals = np.unique(np.round(np.geomspace(32, 10000, 32)).astype(int))
configs = sweep(
                K=100, T=1.0, contract_type="call", S0=100, r=0.05, sigma=0.2,
                s_steps=range(32,2000,128), t_steps=range(32,3000,128),
                version=("v1", "v2", "v3", "v4", "v5", "v6", "v7")
)




from pathlib import Path
df_cn_eu = run_v2(configs=configs, fn=cn_european, n_reps=1, save_dir=None, n_jobs=20, chunksize=64) # paralleled version, set n_jobs based on number of cpu threads
save_to = Path("./data/test_dataset.parquet")
df_cn_eu.to_parquet(save_to)
"""